In [2]:
import os
import sys
import numpy as np
import sklearn.metrics.pairwise
import ot
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans


def get_pred_proba(emb, label_encodings, temperature=1):
    print(f"[get_pred_proba] emb shape: {emb.shape}, label_encodings shape: {label_encodings.shape}")
    sim = sklearn.metrics.pairwise.cosine_similarity(emb, label_encodings)
    M = np.exp(sim / temperature)
    M = M / M.sum(axis=1, keepdims=True)
    return M


def zs_predict(emb, label_encodings, temperature=1):
    pred_proba = get_pred_proba(emb, label_encodings, temperature)
    return pred_proba.argmax(axis=1)


def nlh_cost_matrix(emb, label_encodings, epsilon=1e-9):
    M = get_pred_proba(emb, label_encodings)
    cost = -np.log(M + epsilon)
    return cost


def ot_predict(emb, label_encodings, n_classes, class_balance, epsilon=1e-9):
    src = np.ones(emb.shape[0]) / emb.shape[0]
    tgt = np.ones(n_classes) * class_balance
    print(f"[ot_predict] src sum: {src.sum()}, tgt: {tgt}")
    M = nlh_cost_matrix(emb, label_encodings, epsilon)
    T = ot.emd(src, tgt, M, numItermax=10000000)
    pred = T.argmax(axis=1)
    return pred


def graph_smoothing(T, emb, k=5, smooth_alpha=0.7):
    nbrs = NearestNeighbors(n_neighbors=k, metric='cosine').fit(emb)
    distances, indices = nbrs.kneighbors(emb)
    T_smooth = np.zeros_like(T)
    for i in range(T.shape[0]):
        neighbor_idxs = indices[i]
        T_neighbors_mean = T[neighbor_idxs, :].mean(axis=0)
        T_smooth[i, :] = smooth_alpha * T[i, :] + (1 - smooth_alpha) * T_neighbors_mean
    return T_smooth


def geotter_predict(
    emb,
    label_encodings,
    n_classes,
    class_balance,
    base_temperature=0.1,
    epsilon=1e-9,
    gamma=1.0,
    lambd=1.0,
    use_sinkhorn=False,
    sinkhorn_reg=0.01,
    sinkhorn_iter=100,
    alpha_ot=0.7,
    beta_cluster=0.3,
    cluster_weight=1.0,
    use_smoothing=True,
    neighbor_k=5,
    smooth_alpha=0.7,
):
    print(f"[GeoTTER] emb shape: {emb.shape}, label_encodings shape: {label_encodings.shape}")

    raw_logits = sklearn.metrics.pairwise.cosine_similarity(emb, label_encodings) / base_temperature
    adjusted_logits = gamma * raw_logits + lambd * np.log(class_balance + epsilon)

    max_vals = np.max(adjusted_logits, axis=1, keepdims=True)
    safe_logits = adjusted_logits - max_vals
    exp_logits = np.exp(safe_logits)
    sum_exp = exp_logits.sum(axis=1, keepdims=True)
    adjusted_proba = exp_logits / sum_exp
    cost_ot = -np.log(adjusted_proba + epsilon)

    km = KMeans(n_clusters=n_classes, random_state=42, n_init=10).fit(emb)
    cluster_labels = km.labels_
    cluster_centers = km.cluster_centers_

    cluster_cost = np.zeros_like(cost_ot)
    for i in range(emb.shape[0]):
        center = cluster_centers[cluster_labels[i]].reshape(1, -1)
        sim_vec = sklearn.metrics.pairwise.cosine_similarity(center, label_encodings)[0]
        cluster_cost[i, :] = 1 - sim_vec

    cost_final = alpha_ot * cost_ot + beta_cluster * cluster_cost * cluster_weight

    src = np.ones(emb.shape[0]) / emb.shape[0]
    tgt = class_balance

    if use_sinkhorn:
        T = ot.sinkhorn(src, tgt, cost_final, reg=sinkhorn_reg, numItermax=sinkhorn_iter)
    else:
        T = ot.emd(src, tgt, cost_final, numItermax=10000000)

    if use_smoothing:
        T = graph_smoothing(T, emb, k=neighbor_k, smooth_alpha=smooth_alpha)

    pred = T.argmax(axis=1)
    return pred


def get_class_balance_from_labels(labels, n_classes):
    counts = np.array([(labels == i).sum() for i in range(n_classes)], dtype=np.float64)
    return counts / counts.sum()


def build_dummy_dataset(
    n_samples=120,
    n_classes=3,
    emb_dim=64,
    seed=42,
):
    rng = np.random.default_rng(seed)

    # label text embeddings / class prototypes
    label_encodings = rng.normal(size=(n_classes, emb_dim))
    label_encodings = label_encodings / np.linalg.norm(label_encodings, axis=1, keepdims=True)

    labels = []
    emb_list = []

    samples_per_class = n_samples // n_classes

    for c in range(n_classes):
        center = label_encodings[c]
        class_emb = center + 0.20 * rng.normal(size=(samples_per_class, emb_dim))
        class_emb = class_emb / np.linalg.norm(class_emb, axis=1, keepdims=True)
        emb_list.append(class_emb)
        labels.extend([c] * samples_per_class)

    emb = np.vstack(emb_list)
    labels = np.array(labels)

    remaining = n_samples - len(labels)
    for _ in range(remaining):
        c = rng.integers(0, n_classes)
        center = label_encodings[c]
        sample = center + 0.20 * rng.normal(size=(1, emb_dim))
        sample = sample / np.linalg.norm(sample, axis=1, keepdims=True)
        emb = np.vstack([emb, sample])
        labels = np.append(labels, c)

    return emb, label_encodings, labels


if __name__ == "__main__":
    use_dummy = True

    if use_dummy:
        dataset_name = "DummySet"
        model_name = "DummyBackbone"

        emb, label_encodings, labels = build_dummy_dataset(
            n_samples=120,
            n_classes=3,
            emb_dim=64,
            seed=42,
        )
        n_classes = len(np.unique(labels))
        true_class_balance = get_class_balance_from_labels(labels, n_classes)

    else:
        sys.path.append("../../src")
        import metainfo
        from data_utils import (
            load_emb,
            load_label_encoding,
            load_label,
            get_n_classes,
            get_class_balance,
        )

        base_path = metainfo.test_emb_path
        model_name = "ViT-B/16"
        dataset_name = "Caltech256"

        emb = load_emb(dataset_name=dataset_name, model_name=model_name, base_path=base_path)
        label_encodings = load_label_encoding(dataset_name=dataset_name, model_name=model_name, base_path=base_path)
        labels = load_label(dataset_name=dataset_name, model_name=model_name, base_path=base_path)
        n_classes = get_n_classes(labels)
        true_class_balance = get_class_balance(labels, n_classes)

    print(f"[Main] dataset: {dataset_name}, backbone: {model_name}")
    print(f"[Main] emb shape: {emb.shape}, label_encodings shape: {label_encodings.shape}")

    ot_pred = ot_predict(
        emb,
        label_encodings,
        n_classes=n_classes,
        class_balance=true_class_balance,
    )

    zs_pred = zs_predict(emb, label_encodings)

    geotter_configs = [
        {
            "base_temperature": 0.1,
            "gamma": 1.0,
            "lambd": 1.0,
            "use_sinkhorn": False,
            "alpha_ot": 0.7,
            "beta_cluster": 0.3,
            "cluster_weight": 1.0,
            "use_smoothing": True,
            "neighbor_k": 5,
            "smooth_alpha": 0.7,
        }
    ]

    geotter_results = {}

    for cfg in geotter_configs:
        print("\n==================================================")
        print("GeoTTER parameter combination:", cfg)

        geotter_pred = geotter_predict(
            emb,
            label_encodings,
            n_classes=n_classes,
            class_balance=true_class_balance,
            base_temperature=cfg.get("base_temperature", 0.1),
            gamma=cfg.get("gamma", 1.0),
            lambd=cfg.get("lambd", 1.0),
            use_sinkhorn=cfg.get("use_sinkhorn", False),
            sinkhorn_reg=cfg.get("sinkhorn_reg", 0.01),
            sinkhorn_iter=cfg.get("sinkhorn_iter", 100),
            alpha_ot=cfg.get("alpha_ot", 0.7),
            beta_cluster=cfg.get("beta_cluster", 0.3),
            cluster_weight=cfg.get("cluster_weight", 1.0),
            use_smoothing=cfg.get("use_smoothing", True),
            neighbor_k=cfg.get("neighbor_k", 5),
            smooth_alpha=cfg.get("smooth_alpha", 0.7),
        )

        acc = accuracy_score(labels, geotter_pred)
        dist = np.array([np.sum(geotter_pred == i) for i in range(n_classes)])

        print(f"GeoTTER prediction accuracy: {acc:.4f}")
        print(f"GeoTTER prediction distribution: {dist}")

        geotter_results[str(cfg)] = {
            "accuracy": acc,
            "distribution": dist,
        }

    print("\n=== Accuracy Observation for Each Method ===")
    zs_acc = accuracy_score(labels, zs_pred)
    ot_acc = accuracy_score(labels, ot_pred)

    print(f"ZS Accuracy: {zs_acc:.4f}")
    print(f"Original OT Accuracy: {ot_acc:.4f}")

    print("\n=== Summary of GeoTTER Results for Each Parameter Combination ===")
    for cfg_str, res in geotter_results.items():
        print(f"GeoTTER {cfg_str}: Accuracy = {res['accuracy']:.4f}, Distribution = {res['distribution']}")

    import pandas as pd

    csv_rows = []
    for cfg_str, res in geotter_results.items():
        csv_rows.append(
            {
                "dataset": dataset_name,
                "backbone": model_name,
                "method": "GeoTTER",
                "parameters": cfg_str,
                "accuracy": res["accuracy"],
                "distribution": res["distribution"].tolist(),
            }
        )

    df = pd.DataFrame(csv_rows)
    output_dir = "./GeoTTER_results"
    os.makedirs(output_dir, exist_ok=True)

    output_csv = os.path.join(output_dir, f"{dataset_name}_results.csv")
    df.to_csv(output_csv, index=False)

    print(f"\nResults have been written to CSV file: {output_csv}")

[Main] dataset: DummySet, backbone: DummyBackbone
[Main] emb shape: (120, 64), label_encodings shape: (3, 64)
[ot_predict] src sum: 0.9999999999999999, tgt: [0.33333333 0.33333333 0.33333333]
[get_pred_proba] emb shape: (120, 64), label_encodings shape: (3, 64)
[get_pred_proba] emb shape: (120, 64), label_encodings shape: (3, 64)

GeoTTER parameter combination: {'base_temperature': 0.1, 'gamma': 1.0, 'lambd': 1.0, 'use_sinkhorn': False, 'alpha_ot': 0.7, 'beta_cluster': 0.3, 'cluster_weight': 1.0, 'use_smoothing': True, 'neighbor_k': 5, 'smooth_alpha': 0.7}
[GeoTTER] emb shape: (120, 64), label_encodings shape: (3, 64)
GeoTTER prediction accuracy: 1.0000
GeoTTER prediction distribution: [40 40 40]

=== Accuracy Observation for Each Method ===
ZS Accuracy: 1.0000
Original OT Accuracy: 1.0000

=== Summary of GeoTTER Results for Each Parameter Combination ===
GeoTTER {'base_temperature': 0.1, 'gamma': 1.0, 'lambd': 1.0, 'use_sinkhorn': False, 'alpha_ot': 0.7, 'beta_cluster': 0.3, 'cluster_